# 服の 3D の試作: StdGEN(ステップ 6b)

服を着た素体の絵(正面1枚、背景を透明にしたもの)から、StdGEN(CVPR 2025)で **体・服・髪を分けた 3D** を作る。
もう一つの方法(着せた正面・横・後ろの絵から Hunyuan3D-2mv で作り、体との差で服を切り出す)は、手元とアプリの Colab
サーバーで作るので、このノートブックでは動かさない。計画は docs/model_3d_plan_jp.md のステップ 6b。

- StdGEN のコードは Apache-2.0、学習済みの重みは「研究用」。この試作は、利用者自身の Colab で、利用者の絵だけで動かす
  (公開の Space には絵を送らない)。
- StdGEN は Python 3.9・torch 2.1(CUDA 11.8)を前提にしているので、Colab の Python とは別に uv で 3.9 の環境を作る。

1. ランタイムを **GPU(A100 推奨。L4 でも `LOW_VRAM = True` で動く見込み)** にする。
2. 上から順に実行する。セル「入力」で `stdgen_inputs.zip` を選んでアップロードする。
3. 最後のセルの `stdgen_results.zip` をダウンロードして、アプリの作業者に渡す。

結果は `/content/StdGEN/result/`(A ポーズ・多視点・粗い形・仕上げ)と、時間・VRAM の記録 `/content/stdgen_log.json`。

In [ ]:
# GPU の確認
!nvidia-smi --query-gpu=name,memory.total --format=csv
LOW_VRAM = False  # L4(24GB)で足りなければ True(多視点の段で、モデルを CPU に逃がす。遅くなる)

## 準備(10〜15 分)

In [ ]:
%%bash
set -e
cd /content
pip -q install uv
[ -d StdGEN ] || git clone -q https://github.com/hyz317/StdGEN.git
cd StdGEN
# StdGEN 用の Python 3.9 の環境(Colab の Python とは別)
uv venv -q -p 3.9 /content/venv39
source /content/venv39/bin/activate
uv pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
uv pip install -q xformers==0.0.22.post4 --index-url https://download.pytorch.org/whl/cu118
uv pip install -q torch-scatter -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
uv pip install -q -r requirements.txt
uv pip install -q huggingface_hub
python -c "import torch; print('torch', torch.__version__, torch.cuda.is_available())"

In [ ]:
%%bash
set -e
cd /content/StdGEN
source /content/venv39/bin/activate
mkdir -p ckpt
# 学習済みの重み(研究用)と、SAM(ViT-H)
python -c "from huggingface_hub import snapshot_download; snapshot_download('hyz317/StdGEN', local_dir='ckpt')"
[ -f ckpt/sam_vit_h_4b8939.pth ] || wget -q -O ckpt/sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
ls ckpt

## 入力

`stdgen_inputs.zip`(中に `<名前>.png`。背景を透明にした、服を着た素体の正面の絵)を選ぶ。

In [ ]:
import zipfile, pathlib
from google.colab import files
up = files.upload()
inp = pathlib.Path('/content/StdGEN/input_cases'); inp.mkdir(parents=True, exist_ok=True)
for name in up:
    with zipfile.ZipFile(name) as z:
        z.extractall(inp)
print(sorted(p.name for p in inp.iterdir()))

## 実行(段ごとに時間と GPU メモリを記録する。失敗した段は記録に残して止める)

In [ ]:
import json, subprocess, time, pathlib
py = '/content/venv39/bin/python'
steps = [
    ('apose', ['infer_canonicalize.py', '--input_dir', './input_cases', '--output_dir', './result/apose']),
    ('multiview', ['infer_multiview.py', '--input_dir', './result/apose', '--output_dir', './result/multiview'] + (['--low_vram'] if LOW_VRAM else [])),
    ('slrm', ['infer_slrm.py', '--input_dir', './result/multiview', '--output_dir', './result/slrm']),
    ('refine', ['infer_refine.py', '--input_mv_dir', 'result/multiview', '--input_obj_dir', './result/slrm', '--output_dir', './result/refine']),
]
log = {}
for name, args in steps:
    started = time.time()
    watch = subprocess.Popen(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits', '-lms', '500'],
                             stdout=subprocess.PIPE, text=True)
    run = subprocess.run([py, *args], cwd='/content/StdGEN', capture_output=True, text=True)
    watch.terminate()
    peak = max((int(x) for x in watch.stdout.read().split() if x.isdigit()), default=0)
    log[name] = {'seconds': round(time.time() - started), 'peak_mib': peak, 'ok': run.returncode == 0,
                 'tail': (run.stdout + run.stderr)[-3000:]}
    print(name, log[name]['seconds'], 's', peak, 'MiB', 'ok' if run.returncode == 0 else 'NG')
    if run.returncode != 0:
        print(log[name]['tail'])
        break
pathlib.Path('/content/stdgen_log.json').write_text(json.dumps(log, ensure_ascii=False, indent=1))

In [ ]:
# できたファイルの一覧(体・服・髪が別のファイルか、を見る)
!find /content/StdGEN/result -maxdepth 3 -type f | sort | head -80

## 結果をまとめてダウンロード

In [ ]:
!cd /content && rm -f stdgen_results.zip && zip -qr stdgen_results.zip StdGEN/result stdgen_log.json && ls -lh stdgen_results.zip
from google.colab import files
files.download('/content/stdgen_results.zip')